# Vector Search: FAISS vs Búsqueda Exacta (Brute Force)

Objetivo: comparar latencia y calidad (recall@k) entre búsqueda exacta (NumPy, fuerza bruta) y FAISS (ANN) usando embeddings sintéticos.

Notas técnicas:
- Para simular similitud coseno usamos vectores normalizados y producto interno (FAISS `IndexFlatIP`).
- Si no está disponible `faiss`, el notebook sigue funcionando y mostrará solo la búsqueda exacta.
- Este benchmark es educativo; resultados dependen de hardware y tamaños de N/DIM.

## 1. Generación de Embeddings
Generamos N vectores d-dimensionales normalizados y una consulta.

In [ ]:
import numpy as np, pandas as pd, time
np.random.seed(7)
def make_data(N=10000, DIM=128):
    xb = np.random.randn(N, DIM).astype('float32')
    xb /= np.linalg.norm(xb, axis=1, keepdims=True)
    xq = np.random.randn(DIM).astype('float32')
    xq /= np.linalg.norm(xq)
    return xb, xq
N, DIM, K = 20000, 128, 10
xb, xq = make_data(N, DIM)
ids = np.arange(N)
N, DIM, K

## 2. Búsqueda Exacta (Brute Force)
Calculamos `scores = xb @ xq` (dot product) y obtenemos los top‑K.

In [ ]:
def topk_bruteforce(xb: np.ndarray, xq: np.ndarray, k: int):
    t0 = time.perf_counter()
    scores = xb @ xq
    idx = np.argpartition(scores, -k)[-k:]
    top = idx[np.argsort(scores[idx])[::-1]]
    latency_ms = (time.perf_counter() - t0)*1000
    return top, scores[top], latency_ms
bf_idx, bf_scores, bf_ms = topk_bruteforce(xb, xq, K)
bf_idx[:5], bf_scores[:5], bf_ms

## 3. FAISS (IndexFlatIP)
Construimos un índice de producto interno para aproximar similitud coseno (con vectores normalizados).

In [ ]:
try:
    import faiss  # faiss-cpu
    index = faiss.IndexFlatIP(DIM)
    # Añadimos embeddings (ya normalizados)
    index.add(xb)
    t0 = time.perf_counter()
    D, I = index.search(xq.reshape(1, -1), K)
    faiss_ms = (time.perf_counter() - t0)*1000
    faiss_idx = I[0]
    faiss_scores = D[0]
except Exception as e:
    faiss = None
    faiss_idx, faiss_scores, faiss_ms = None, None, None
    print('FAISS no disponible o error al importar:', e)
faiss_idx, faiss_scores, faiss_ms

## 4. Métrica de calidad: Recall@K
Recall@K = |TopK_FAISS ∩ TopK_Exacto| / K.

In [ ]:
def recall_at_k(bf_idx, faiss_idx, k):
    if faiss_idx is None:
        return np.nan
    return len(set(bf_idx.tolist()) & set(faiss_idx.tolist()))/k
recall = recall_at_k(bf_idx, faiss_idx, K)
summary = pd.Series({
    'N': N, 'DIM': DIM, 'K': K,
    'bf_latency_ms': bf_ms,
    'faiss_latency_ms': faiss_ms,
    'recall@K': recall
})
summary

## 5. Experimento opcional: barrer tamaños
Medimos latencia vs N para BF y FAISS (si disponible).

In [ ]:
def bench_sweep(N_list=(5000, 10000, 20000), dim=128, k=10):
    rows = []
    for n in N_list:
        xb, xq = make_data(n, dim)
        bf_idx, _, bf_ms = topk_bruteforce(xb, xq, k)
        faiss_ms_val, rec_val = np.nan, np.nan
        if 'faiss' in globals() and faiss is not None:
            index = faiss.IndexFlatIP(dim)
            index.add(xb)
            t0 = time.perf_counter()
            D, I = index.search(xq.reshape(1, -1), k)
            faiss_ms_val = (time.perf_counter() - t0)*1000
            rec_val = recall_at_k(bf_idx, I[0], k)
        rows.append({'N': n, 'bf_ms': bf_ms, 'faiss_ms': faiss_ms_val, 'recall@k': rec_val})
    return pd.DataFrame(rows)
bench_sweep()

## 6. Conclusiones
- `IndexFlatIP` (FAISS) ofrece latencias menores frente a BF para consultas, con recall=1.0 (índice exacto).
- Para índices aproximados (IVF/HNSW/PQ) el trade-off latencia/recall depende de configuración.
- En producción, combinar ANN + filtros por payload y re-ranking de precisión.

**Referencias:** FAISS (Facebook AI Research) https://github.com/facebookresearch/faiss